# 도로교통법 전문 상담 AI Agent

이 노트북은 `./common_dataset`의 다음 자료만 사용해 처음부터 구축합니다.

- `도로교통법_법률_제21246호.pdf` — **2026. 7. 1. 시행** 도로교통법
- `판례목록.csv` — 도로교통법 관련 판례 요약

법령과 판례를 별도 인덱스로 검색하고, 벡터 검색과 키워드 검색을 결합한 하이브리드 RAG를 LangGraph 상담 워크플로에 연결합니다.

> ⚠️ 이 챗봇은 교육·정보 제공용이며 변호사의 법률 자문을 대체하지 않습니다. 실제 처분은 시행령·시행규칙, 사고 증거, 전과, 관할기관 판단 등에 따라 달라질 수 있습니다.

In [ ]:
# 최초 1회 실행: 설치 후 커널을 재시작하고 다음 셀부터 실행하세요.
# %pip install -q -U langchain langchain-openai langchain-community langchain-chroma langgraph chromadb pypdf pandas rank-bm25 gradio python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [16]:
from __future__ import annotations

import getpass
import hashlib
import os
import re
from pathlib import Path
from typing import Annotated, Literal, TypedDict

import pandas as pd
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from pypdf import PdfReader
from rank_bm25 import BM25Okapi

load_dotenv()

# OpenAI 외 모델을 쓰려면 이 두 객체만 호환 구현으로 교체하면 됩니다.
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

CHAT_MODEL = os.getenv("TRAFFIC_LAW_CHAT_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("TRAFFIC_LAW_EMBEDDING_MODEL", "text-embedding-3-small")
llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

print("chat model:", CHAT_MODEL)
print("embedding model:", EMBEDDING_MODEL)

chat model: gpt-4.1-mini
embedding model: text-embedding-3-small


In [17]:
# Jupyter의 실행 위치가 노트북 폴더 또는 프로젝트 루트인 경우를 모두 지원합니다.
data_candidates = [Path("./common_dataset"), Path("./doitmyself/common_dataset")]
DATA_DIR = next((p.resolve() for p in data_candidates if p.exists()), None)
if DATA_DIR is None:
    tried = ", ".join(str(p.resolve()) for p in data_candidates)
    raise FileNotFoundError(f"common_dataset 폴더를 찾을 수 없습니다. 확인한 위치: {tried}")

LAW_PDF = DATA_DIR / "도로교통법_법률_제21246호.pdf"
CASE_CSV = DATA_DIR / "판례목록.csv"
DB_DIR = DATA_DIR.parent / ".traffic_law_z_chroma"

for path in (LAW_PDF, CASE_CSV):
    if not path.exists():
        raise FileNotFoundError(f"데이터 파일을 찾을 수 없습니다: {path}")

print("data:", DATA_DIR)
print("law PDF:", LAW_PDF.name)
print("case CSV:", CASE_CSV.name)

data: D:\Project\AI_class\doitmyself\common_dataset
law PDF: 도로교통법_법률_제21246호.pdf
case CSV: 판례목록.csv


In [18]:
ARTICLE_RE = re.compile(r"제\s*(\d+)\s*조(?:의\s*(\d+))?(?:\([^)]*\))?")
TOKEN_RE = re.compile(r"제\d+조(?:의\d+)?|[가-힣A-Za-z]+|\d+(?:\.\d+)?")


def article_label(match: re.Match) -> str:
    return f"제{match.group(1)}조" + (f"의{match.group(2)}" if match.group(2) else "")


def compact_text(text: str) -> str:
    """PDF 줄바꿈/공백을 검색에 적합하게 정리합니다."""
    text = text.replace("\x00", " ")
    text = re.sub(r"--\s*\d+\s*of\s*\d+\s*--", " ", text, flags=re.I)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_text(text: str, chunk_size: int = 1100, overlap: int = 160) -> list[str]:
    """문장 경계를 우선 보존하는 간단하고 재현 가능한 청킹입니다."""
    if len(text) <= chunk_size:
        return [text]
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            boundary = max(text.rfind(". ", start, end), text.rfind("다. ", start, end))
            if boundary > start + chunk_size // 2:
                end = boundary + 2
        chunks.append(text[start:end].strip())
        if end == len(text):
            break
        start = max(start + 1, end - overlap)
    return [chunk for chunk in chunks if chunk]


def load_law_documents(pdf_path: Path) -> list[Document]:
    reader = PdfReader(str(pdf_path))
    documents: list[Document] = []
    current_article = "미상"
    for page_no, page in enumerate(reader.pages, start=1):
        text = compact_text(page.extract_text() or "")
        if not text:
            continue
        matches = list(ARTICLE_RE.finditer(text))
        if matches:
            current_article = article_label(matches[0])
        for chunk_no, chunk in enumerate(split_text(text)):
            chunk_articles = [article_label(m) for m in ARTICLE_RE.finditer(chunk)]
            article = ", ".join(dict.fromkeys(chunk_articles)) or current_article
            documents.append(Document(
                page_content=chunk,
                metadata={
                    "source_type": "법령",
                    "source": pdf_path.name,
                    "page": page_no,
                    "article": article,
                    "chunk": chunk_no,
                    "effective_date": "2026-07-01",
                },
            ))
        if matches:
            current_article = article_label(matches[-1])
    return documents


def load_case_documents(csv_path: Path) -> list[Document]:
    frame = pd.read_csv(csv_path, encoding="utf-8-sig").fillna("")
    required = {"제목", "사건번호", "선고일자", "조문번호", "판시사항"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"판례 CSV 필수 열 누락: {sorted(missing)}")

    documents: list[Document] = []
    for row_no, row in frame.iterrows():
        content = (
            f"사건명: {row['제목']}\n사건번호: {row['사건번호']}\n"
            f"선고일자: {row['선고일자']}\n관련 조문: {row['조문번호']}\n"
            f"판시사항: {row['판시사항']}"
        ).strip()
        documents.append(Document(
            page_content=content,
            metadata={
                "source_type": "판례",
                "source": csv_path.name,
                "row": int(row_no) + 2,
                "case_name": str(row["제목"]),
                "case_number": str(row["사건번호"]),
                "decision_date": str(row["선고일자"]),
                "articles": str(row["조문번호"]),
            },
        ))
    return documents


law_documents = load_law_documents(LAW_PDF)
case_documents = load_case_documents(CASE_CSV)
print(f"법령 청크: {len(law_documents):,}개 / 판례: {len(case_documents):,}건")

법령 청크: 121개 / 판례: 410건


In [19]:
def dataset_fingerprint(*paths: Path) -> str:
    digest = hashlib.sha256()
    for path in paths:
        digest.update(path.name.encode("utf-8"))
        digest.update(str(path.stat().st_size).encode())
        digest.update(str(path.stat().st_mtime_ns).encode())
    return digest.hexdigest()[:12]


def stable_id(doc: Document) -> str:
    raw = f"{doc.metadata}|{doc.page_content}".encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


fingerprint = dataset_fingerprint(LAW_PDF, CASE_CSV)
DB_DIR.mkdir(parents=True, exist_ok=True)

law_store = Chroma(
    collection_name=f"traffic_law_{fingerprint}",
    embedding_function=embeddings,
    persist_directory=str(DB_DIR),
)
case_store = Chroma(
    collection_name=f"traffic_cases_{fingerprint}",
    embedding_function=embeddings,
    persist_directory=str(DB_DIR),
)

# 같은 데이터로 셀을 재실행할 때 중복 임베딩하지 않습니다.
if law_store._collection.count() == 0:
    law_store.add_documents(law_documents, ids=[stable_id(d) for d in law_documents])
if case_store._collection.count() == 0:
    case_store.add_documents(case_documents, ids=[stable_id(d) for d in case_documents])

print("법령 벡터:", law_store._collection.count())
print("판례 벡터:", case_store._collection.count())

법령 벡터: 121
판례 벡터: 410


In [20]:
def tokenize(text: str) -> list[str]:
    return [token.lower() for token in TOKEN_RE.findall(text)]


class HybridRetriever:
    """의미 검색과 BM25 검색 순위를 RRF(Reciprocal Rank Fusion)로 결합합니다."""

    def __init__(self, documents: list[Document], vector_store: Chroma):
        self.documents = documents
        self.vector_store = vector_store
        self.bm25 = BM25Okapi([tokenize(d.page_content) for d in documents])

    def search(self, query: str, k: int = 6, fetch_k: int = 12) -> list[Document]:
        vector_hits = self.vector_store.similarity_search(query, k=min(fetch_k, len(self.documents)))
        keyword_scores = self.bm25.get_scores(tokenize(query))
        keyword_indices = sorted(
            range(len(keyword_scores)), key=lambda i: keyword_scores[i], reverse=True
        )[:fetch_k]
        keyword_hits = [self.documents[i] for i in keyword_indices if keyword_scores[i] > 0]

        fused: dict[str, tuple[float, Document]] = {}
        for weight, hits in ((1.0, vector_hits), (1.15, keyword_hits)):
            for rank, doc in enumerate(hits, start=1):
                key = stable_id(doc)
                score = weight / (60 + rank)
                old_score = fused.get(key, (0.0, doc))[0]
                fused[key] = (old_score + score, doc)
        return [item[1] for item in sorted(fused.values(), key=lambda x: x[0], reverse=True)[:k]]


law_retriever = HybridRetriever(law_documents, law_store)
case_retriever = HybridRetriever(case_documents, case_store)


def source_label(doc: Document) -> str:
    meta = doc.metadata
    if meta["source_type"] == "법령":
        return f"법령 {meta.get('article', '조문 미상')} (PDF p.{meta['page']})"
    return f"판례 {meta.get('case_number', '사건번호 미상')} ({meta.get('decision_date', '선고일 미상')})"


def format_context(law_hits: list[Document], case_hits: list[Document]) -> str:
    blocks: list[str] = []
    for i, doc in enumerate(law_hits, start=1):
        blocks.append(f"[L{i} | {source_label(doc)}]\n{doc.page_content}")
    for i, doc in enumerate(case_hits, start=1):
        blocks.append(f"[C{i} | {source_label(doc)}]\n{doc.page_content}")
    return "\n\n".join(blocks)

# 검색 동작 확인
for doc in law_retriever.search("음주운전 혈중알코올농도 벌금 처벌", k=3):
    print(source_label(doc))

법령 제50조의2, 제50조의3, 제80조의2, 제82조 (PDF p.27)
법령 제50조의2 (PDF p.27)
법령 제148조의2, 제148조의3, 제50조의3 (PDF p.45)


## Agent 워크플로

1. **사실관계·쟁점 분석**: 질문에서 사고 유형, 행위, 피해, 음주 수치, 면허, 사후조치 등 핵심 요소를 구조화합니다.
2. **이중 검색**: 현행 법령과 판례를 각각 벡터 + BM25로 검색합니다.
3. **근거 제한 답변**: 검색된 문서에 있는 내용만 인용하고, 부족한 사실과 추가 확인 법령을 분리합니다.
4. **대화 메모리**: 같은 `thread_id`에서 후속 질문의 맥락을 유지합니다.

In [21]:
class LegalIssueAnalysis(BaseModel):
    category: Literal[
        "교통사고", "음주·약물", "무면허", "속도·신호", "주정차",
        "개인형이동장치", "면허행정", "기타"
    ] = Field(description="가장 중요한 상담 유형")
    issues: list[str] = Field(description="검토할 법적 쟁점")
    search_queries: list[str] = Field(description="법령·판례 검색용 한국어 질의 1~3개")
    likely_articles: list[str] = Field(description="질문에서 추정되는 조문 번호. 모르면 빈 목록")
    missing_facts: list[str] = Field(description="판단에 중요하지만 질문에 없는 사실")


class AgentState(TypedDict, total=False):
    messages: Annotated[list[BaseMessage], add_messages]
    analysis: dict
    law_hits: list[Document]
    case_hits: list[Document]
    context: str


ANALYSIS_SYSTEM = """당신은 대한민국 도로교통 사건의 쟁점을 구조화하는 분석 Agent다.
사용자의 최신 질문과 대화 맥락에서 검색에 필요한 사실만 추출한다.
검색 질의에는 구체적인 행위, 결과, 관련 조문 후보, 처벌/행정처분 쟁점을 포함한다.
없는 사실을 추정하지 말고 missing_facts에 기록한다."""

ANSWER_SYSTEM = """당신은 대한민국 도로교통법 전문 상담 AI다.
아래 원칙을 반드시 지켜라.

- 제공된 검색 근거만으로 답하고 법령 문구, 벌금액, 판례 결론을 만들지 않는다.
- 현행법 기준일은 2026-07-01(법률 제21246호)임을 명시한다.
- 먼저 '잠정 판단'을 짧게 말하고, 그 다음 '적용 근거', '추가로 필요한 사실', '권장 조치' 순서로 쓴다.
- 모든 핵심 법적 판단 문장 끝에 [L1], [C1]처럼 실제 제공된 근거 ID를 붙인다.
- 법령과 판례가 충돌하거나 판례가 과거 법령을 다루면 현행 법령을 우선하고 시점 차이를 경고한다.
- 자료에 시행령·시행규칙·교통사고처리특례법·특정범죄가중처벌법 등이 없으면 해당 법률의 확인 필요성을 분명히 밝힌다.
- 사실이 부족하면 단정하지 말고 조건부로 설명한 뒤 필요한 질문을 최대 5개 제시한다.
- 사고 직후 상황이면 인명 구조, 119·112 신고, 정차, 피해자 구호, 위험 방지, 인적사항 제공을 우선 안내한다.
- 최종 형량·과실비율·유무죄를 확정하지 않는다.
- 마지막에 '이 답변은 제공 자료에 기초한 일반 정보이며 법률 자문이 아닙니다.'라고 쓴다.
"""

structured_analyzer = llm.with_structured_output(LegalIssueAnalysis)


def latest_user_text(messages: list[BaseMessage]) -> str:
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return str(message.content)
    raise ValueError("사용자 질문이 없습니다.")


def analyze_node(state: AgentState) -> dict:
    recent = state["messages"][-6:]
    analysis = structured_analyzer.invoke([
        SystemMessage(content=ANALYSIS_SYSTEM),
        *recent,
    ])
    return {"analysis": analysis.model_dump()}


def retrieve_node(state: AgentState) -> dict:
    user_question = latest_user_text(state["messages"])
    analysis = LegalIssueAnalysis.model_validate(state["analysis"])
    queries = analysis.search_queries or [user_question]
    if analysis.likely_articles:
        queries.append(" ".join(analysis.likely_articles + analysis.issues))

    def collect(retriever: HybridRetriever, limit: int) -> list[Document]:
        unique: dict[str, Document] = {}
        for query in queries[:4]:
            for doc in retriever.search(query, k=limit):
                unique.setdefault(stable_id(doc), doc)
        return list(unique.values())[:limit]

    law_hits = collect(law_retriever, 7)
    case_hits = collect(case_retriever, 5)
    return {
        "law_hits": law_hits,
        "case_hits": case_hits,
        "context": format_context(law_hits, case_hits),
    }


def answer_node(state: AgentState) -> dict:
    analysis = LegalIssueAnalysis.model_validate(state["analysis"])
    prompt = f"""[쟁점 분석]
{analysis.model_dump_json(indent=2)}

[검색 근거]
{state['context']}

[사용자 질문]
{latest_user_text(state['messages'])}

검색 근거의 ID를 정확히 인용해 한국어로 답하라."""
    response = llm.invoke([
        SystemMessage(content=ANSWER_SYSTEM),
        *state["messages"][-6:-1],
        HumanMessage(content=prompt),
    ])
    return {"messages": [response]}


builder = StateGraph(AgentState)
builder.add_node("analyze", analyze_node)
builder.add_node("retrieve", retrieve_node)
builder.add_node("answer", answer_node)
builder.add_edge(START, "analyze")
builder.add_edge("analyze", "retrieve")
builder.add_edge("retrieve", "answer")
builder.add_edge("answer", END)

agent = builder.compile(checkpointer=MemorySaver())
print("Agent graph ready")

Agent graph ready


In [22]:
def consult(question: str, thread_id: str = "traffic-law-demo", show_sources: bool = False) -> str:
    """상담 질문을 실행합니다. 같은 thread_id는 후속 대화 맥락을 공유합니다."""
    if not question or not question.strip():
        return "질문을 입력해 주세요."
    result = agent.invoke(
        {"messages": [HumanMessage(content=question.strip())]},
        config={"configurable": {"thread_id": thread_id}},
    )
    answer = str(result["messages"][-1].content)
    if show_sources:
        labels = [source_label(d) for d in result.get("law_hits", []) + result.get("case_hits", [])]
        answer += "\n\n---\n검색된 자료\n" + "\n".join(f"- {label}" for label in labels)
    return answer


# 첫 상담 예시: 실행 시 OpenAI API 호출이 발생합니다.
print(consult(
    "주차된 차를 살짝 긁었는데 차주가 없어서 메모를 남기지 않고 왔습니다. "
    "사고 후 미조치로 처벌될 수 있나요?",
    thread_id="example-1",
    show_sources=True,
))

잠정 판단:  
주차된 차량에 경미한 접촉 사고가 발생한 경우에도 도로교통법 제54조에 따라 사고 발생 시 운전자는 즉시 정차하여 피해자 구호 및 인적 사항 제공 등 필요한 조치를 해야 하며, 사고 후 조치를 하지 않고 현장을 이탈하면 도로교통법 제148조에 따라 처벌될 수 있습니다. 따라서 차주가 없다고 하여 메모를 남기지 않고 떠난 경우에도 처벌 가능성이 있습니다[L1][C1].

적용 근거:  
- 도로교통법 제54조(사고발생 시의 조치)에서는 교통사고로 물건을 손괴한 경우에도 운전자가 즉시 정차하여 피해자에게 인적 사항을 제공하고 필요한 조치를 하도록 규정하고 있습니다.  
- 제54조 제2항은 경찰공무원이 현장에 없을 때에는 가장 가까운 경찰관서에 사고 사실을 신고하도록 명시하고 있습니다.  
- 도로교통법 제148조는 사고 후 조치 의무 위반에 대한 처벌 근거입니다.  
- 판례(2010도1330)는 사고 후 피해자 구호 등의 조치를 취할 필요가 있다고 판단한 사례로, 사고 사실을 인식한 상태에서 조치를 하지 않고 현장을 이탈하면 처벌될 수 있음을 보여줍니다[C1].

추가로 필요한 사실:  
1. 사고 당시 차량 파손 정도는 어느 정도였는지?  
2. 사고 발생 장소와 시간은 언제 어디였는지?  
3. 사고 후 현장에 머문 시간은 얼마나 되었는지?  
4. 메모를 남기지 않고 떠난 이유가 무엇인지?  
5. 피해 차량 소유자와 연락을 시도했는지 여부?

권장 조치:  
- 사고 발생 시 즉시 정차하여 피해 차량의 상태를 확인하고, 피해자에게 인적 사항을 남기거나 연락처를 남기는 것이 법적 의무입니다.  
- 차주가 현장에 없을 경우에는 메모를 남기고, 가까운 경찰서에 사고 사실을 신고하는 것이 바람직합니다.  
- 향후 유사 사고 방지를 위해 사고 발생 시 반드시 현장에 머무르며 필요한 조치를 취할 것을 권장합니다.

현행법 기준일은 2026-07-01(법률 제21246호)임을 명시합니다.  
이 답변은 제공 자료에 기초한 일반 정보이며 법률 자문

In [26]:
# 선택 실행: 브라우저 기반 채팅 UI
import uuid
import gradio as gr


def gradio_chat(message: str, history: list, thread_id: str) -> str:
    return consult(message, thread_id=thread_id, show_sources=False)


# additional_inputs를 쓰면 examples는 [[메시지, additional_input...], ...] 형식이어야 합니다.
# State 값은 예시에서 바꾸지 않으므로 None을 둡니다.
thread_state = gr.State(str(uuid.uuid4()))
demo = gr.ChatInterface(
    fn=gradio_chat,
    additional_inputs=[thread_state],
    title="도로교통법 전문 상담 AI Agent",
    description=(
        "2026. 7. 1. 시행 도로교통법과 제공된 판례 요약을 검색해 답합니다. "
        "개인정보(이름, 차량번호, 연락처)는 입력하지 마세요."
    ),
    examples=[
        ["혈중알코올농도 0.05% 상태로 운전하면 어떤 처벌을 받을 수 있나요?", None],
        ["신호 없는 횡단보도에서 보행자와 사고가 났습니다. 어떤 사실을 확인해야 하나요?", None],
        ["전동킥보드를 술을 마시고 타면 자동차 음주운전과 처벌이 같나요?", None],
        ["면허 없이 아파트 주차장에서 운전한 경우에도 무면허운전인가요?", None],
    ],
)

demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# Gold test set 순차 평가 (Agent 답변 + LLM-as-a-Judge, 문항당 10점)
# 실행 시 문항마다 Agent 2회(쟁점 분석/답변) + Judge 1회의 API 호출이 발생합니다.
import time
import uuid
from IPython.display import display
from pydantic import BaseModel, Field


TEST_DIR_CANDIDATES = [Path("./test_dataset"), Path("./doitmyself/test_dataset")]
TEST_DIR = next((p.resolve() for p in TEST_DIR_CANDIDATES if p.exists()), None)
if TEST_DIR is None:
    tried = ", ".join(str(p.resolve()) for p in TEST_DIR_CANDIDATES)
    raise FileNotFoundError(f"test_dataset 폴더를 찾을 수 없습니다. 확인한 위치: {tried}")

GOLD_CSV = TEST_DIR / "road_traffic_law_gold_test_set_50.csv"
gold_df = pd.read_csv(GOLD_CSV, encoding="utf-8-sig").fillna("")
required_columns = {
    "id", "question", "difficulty", "required_articles", "legal_issue",
    "key_facts", "missing_facts", "expected_conclusion", "should_abstain",
    "must_not_claim", "reference_date",
}
missing_columns = required_columns - set(gold_df.columns)
if missing_columns:
    raise ValueError(f"Gold test set 필수 열 누락: {sorted(missing_columns)}")


# 테스트 데이터 선택 설정
TEST_DATA_COUNT = 50          # 사용할 데이터 수 (int)
TEST_DIFFICULTY = "easy"      # "even", "easy"(5:3:2), "low", "medium", "hard"
TEST_EXTRACTION_METHOD = "ascend"  # "rand", "ascend", "descend"
TEST_RANDOM_SEED = 42         # rand 추출 결과 재현용

# CSV에 easy/low가 섞여 있어도 동일 난이도로 정규화합니다.
DIFFICULTY_ALIASES = {
    "easy": "low",
    "low": "low",
    "medium": "medium",
    "hard": "hard",
}
# TEST_DIFFICULTY="easy"일 때 low:medium:hard 추출 비율
EASY_MIX_RATIO = {"low": 5, "medium": 3, "hard": 2}


def allocate_by_ratio(total: int, ratios: dict[str, int]) -> dict[str, int]:
    """총 개수를 비율에 맞게 정수로 배분합니다. 남는 수는 소수부가 큰 난이도부터 줍니다."""
    parts = sum(ratios.values())
    raw = {level: total * weight / parts for level, weight in ratios.items()}
    counts = {level: int(value) for level, value in raw.items()}
    remainder = total - sum(counts.values())
    ordered = sorted(
        ratios,
        key=lambda level: (raw[level] - counts[level], ratios[level]),
        reverse=True,
    )
    for level in ordered[:remainder]:
        counts[level] += 1
    return counts


def select_test_data(
    tests: pd.DataFrame,
    data_count: int,
    difficulty: str,
    extraction_method: str,
    random_seed: int | None = None,
) -> pd.DataFrame:
    """설정에 따라 평가에 사용할 테스트 데이터 일부를 선택합니다."""
    valid_difficulties = {"even", "easy", "low", "medium", "hard"}
    valid_methods = {"rand", "ascend", "descend"}

    if not isinstance(data_count, int) or isinstance(data_count, bool) or data_count <= 0:
        raise ValueError("data_count는 1 이상의 int여야 합니다.")
    if difficulty not in valid_difficulties:
        raise ValueError(f"difficulty는 {sorted(valid_difficulties)} 중 하나여야 합니다.")
    if extraction_method not in valid_methods:
        raise ValueError(f"extraction_method는 {sorted(valid_methods)} 중 하나여야 합니다.")

    normalized = tests.copy()
    normalized["difficulty"] = (
        normalized["difficulty"]
        .astype(str)
        .str.lower()
        .str.strip()
        .map(lambda value: DIFFICULTY_ALIASES.get(value, value))
    )

    def extract(frame: pd.DataFrame, count: int, seed_offset: int = 0) -> pd.DataFrame:
        if count <= 0:
            return frame.iloc[0:0].copy()
        if len(frame) < count:
            available = sorted(normalized["difficulty"].unique())
            raise ValueError(
                f"조건에 맞는 데이터가 부족합니다: 필요 {count}개, 사용 가능 {len(frame)}개 "
                f"(현재 difficulty 값: {available})"
            )
        if extraction_method == "rand":
            seed = None if random_seed is None else random_seed + seed_offset
            return frame.sample(n=count, random_state=seed)

        ascending = extraction_method == "ascend"
        return frame.sort_values("id", ascending=ascending).head(count)

    def extract_by_counts(counts: dict[str, int]) -> pd.DataFrame:
        selected_parts = [
            extract(
                normalized[normalized["difficulty"] == level],
                count,
                seed_offset=index,
            )
            for index, (level, count) in enumerate(counts.items())
            if count > 0
        ]
        if not selected_parts:
            raise ValueError("추출할 데이터가 없습니다.")
        selected = pd.concat(selected_parts)
        if extraction_method != "rand":
            selected = selected.sort_values(
                "id", ascending=extraction_method == "ascend"
            )
        return selected

    if difficulty == "even":
        levels = ["low", "medium", "hard"]
        present_levels = [level for level in levels if (normalized["difficulty"] == level).any()]
        if not present_levels:
            raise ValueError("even 추출에 사용할 난이도(low/medium/hard) 데이터가 없습니다.")
        base, remainder = divmod(data_count, len(present_levels))
        counts = {
            level: base + (index < remainder)
            for index, level in enumerate(present_levels)
        }
        selected = extract_by_counts(counts)
    elif difficulty == "easy":
        # easy = 혼합 추출 모드 (low:medium:hard = 5:3:2)
        counts = allocate_by_ratio(data_count, EASY_MIX_RATIO)
        selected = extract_by_counts(counts)
    else:
        selected = extract(normalized[normalized["difficulty"] == difficulty], data_count)

    return selected.reset_index(drop=True)


selected_gold_df = select_test_data(
    gold_df,
    data_count=TEST_DATA_COUNT,
    difficulty=TEST_DIFFICULTY,
    extraction_method=TEST_EXTRACTION_METHOD,
    random_seed=TEST_RANDOM_SEED,
)
print(
    f"선택된 테스트 데이터: {len(selected_gold_df)}개 | "
    f"난이도별 {selected_gold_df['difficulty'].value_counts().to_dict()}"
)
display(selected_gold_df[["id", "difficulty", "question"]])


class GoldJudgeResult(BaseModel):
    conclusion_score: int = Field(ge=0, le=4, description="기대 결론과의 일치도")
    article_score: int = Field(ge=0, le=2, description="필수 조문의 정확한 적용")
    abstention_score: int = Field(ge=0, le=2, description="불충분한 사실에 대한 조건부 판단")
    adopted_forbidden_claims: list[str] = Field(
        description=(
            "Agent가 잠정 판단·적용 근거에서 자신의 결론으로 채택하거나 "
            "실질적으로 지지한 must_not_claim 항목. 해당 없으면 빈 목록."
        )
    )
    ambiguous_forbidden_claims: list[str] = Field(
        description=(
            "명시적으로 부정하지도, 명확히 채택하지도 않아 금지 주장 회피가 "
            "불분명한 must_not_claim 항목. 해당 없으면 빈 목록."
        )
    )
    rejected_forbidden_claims: list[str] = Field(
        description=(
            "Agent가 명시적으로 부정했거나 정반대 결론을 내어 "
            "회피했다고 볼 수 있는 must_not_claim 항목."
        )
    )
    passed: bool = Field(description="중대한 법적 오류 없이 실용적으로 적절한 답변인지")
    rationale: str = Field(description="점수 근거를 3문장 이내로 설명")


JUDGE_SYSTEM = """당신은 대한민국 도로교통법 QA 평가자다.
Gold 기준을 유일한 정답 기준으로 삼아 Agent 답변의 법적 판단을 엄격히 채점한다.
표현이 달라도 의미가 같으면 인정하되, Gold에 없는 내용을 임의로 정답 처리하지 않는다.

채점 기준(총점은 코드에서 합산):
- conclusion_score 0~4: 기대 결론과 핵심 법적 판단의 일치도
- article_score 0~2: required_articles를 정확히 언급하거나 그 규범 내용을 정확히 적용했는지
- abstention_score 0~2: should_abstain과 missing_facts에 맞게 단정/조건부 판단을 했는지

금지 주장(must_not_claim) 분류 규칙:
1. must_not_claim은 ';'로 구분된 금지 주장 목록이다. 각 항목을 빠짐없이 검토한다.
2. Agent가 그 내용을 자신의 결론·판단으로 채택하거나 실질적으로 지지하면 adopted_forbidden_claims에 넣는다.
3. 명시적으로 부정하거나 정반대 결론이면 rejected_forbidden_claims에 넣는다.
4. 언급도 부정도 채택도 불분명하면 ambiguous_forbidden_claims에 넣는다.
5. 금지 주장을 인용만 하고 명시적으로 부정한 경우는 rejected로 본다.
6. adopted / ambiguous / rejected에 같은 항목을 중복하지 않는다.
7. 금지 주장 회피 점수는 코드가 adopted/ambiguous 목록으로 계산하므로 정수 점수를 직접 쓰지 않는다.

passed는 conclusion_score가 3점 이상이고, adopted_forbidden_claims가 비어 있으며, 실용적으로 적절한 경우에만 true다."""

judge = llm.with_structured_output(GoldJudgeResult)


def parse_must_not_claims(raw: str) -> list[str]:
    return [part.strip() for part in str(raw).split(";") if part.strip()]


def score_forbidden_claims(
    adopted: list[str],
    ambiguous: list[str],
) -> int:
    """금지 주장 회피 점수를 목록에서 확정합니다. LLM 정수 필드를 신뢰하지 않습니다."""
    if adopted:
        return 0
    if ambiguous:
        return 1
    return 2


def evaluate_gold_test_set(
    tests: pd.DataFrame,
    delay_seconds: float = 0.2,
    save_csv: bool = True,
) -> pd.DataFrame:
    """Gold 질문을 순차 실행하고 문항별 10점 만점으로 평가합니다."""
    run_id = uuid.uuid4().hex[:8]
    records: list[dict] = []

    for index, row in tests.reset_index(drop=True).iterrows():
        test_id = str(row["id"])
        print(f"[{index + 1}/{len(tests)}] {test_id} Agent 질의 중...", flush=True)

        answer_started_at = time.perf_counter()
        response_seconds: float | None = None
        try:
            # 문항 간 대화 내용이 섞이지 않도록 매번 고유 thread_id를 사용합니다.
            answer = consult(
                str(row["question"]),
                thread_id=f"gold-{run_id}-{test_id}",
                show_sources=False,
            )
            # Judge 채점 시간은 제외하고 Agent가 답변을 반환할 때까지만 측정합니다.
            response_seconds = round(time.perf_counter() - answer_started_at, 3)

            must_not_claims = parse_must_not_claims(row["must_not_claim"])
            gold_context = f"""[Gold]
id: {test_id}
question: {row['question']}
difficulty: {row['difficulty']}
required_articles: {row['required_articles']}
legal_issue: {row['legal_issue']}
key_facts: {row['key_facts']}
missing_facts: {row['missing_facts']}
expected_conclusion: {row['expected_conclusion']}
should_abstain: {row['should_abstain']}
must_not_claim 목록:
{chr(10).join(f'- {claim}' for claim in must_not_claims) or '- (없음)'}
reference_date: {row['reference_date']}

[Agent 답변]
{answer}

위 must_not_claim 목록의 각 항목을 adopted / ambiguous / rejected 중 하나에만 분류하라."""
            grade = judge.invoke([
                SystemMessage(content=JUDGE_SYSTEM),
                HumanMessage(content=gold_context),
            ])
            # 금지 주장 점수는 LLM 정수 필드가 아니라 분류 목록에서 확정합니다.
            forbidden_claim_score = score_forbidden_claims(
                grade.adopted_forbidden_claims,
                grade.ambiguous_forbidden_claims,
            )
            total_score = (
                grade.conclusion_score
                + grade.article_score
                + grade.abstention_score
                + forbidden_claim_score
            )
            passed = bool(
                total_score >= 7
                and grade.conclusion_score >= 3
                and forbidden_claim_score > 0
                and not grade.adopted_forbidden_claims
                and grade.passed
            )
            records.append({
                "id": test_id,
                "difficulty": row["difficulty"],
                "question": row["question"],
                "answer": answer,
                "response_seconds": response_seconds,
                "conclusion_score": grade.conclusion_score,
                "article_score": grade.article_score,
                "abstention_score": grade.abstention_score,
                "forbidden_claim_score": forbidden_claim_score,
                "adopted_forbidden_claims": " | ".join(grade.adopted_forbidden_claims),
                "ambiguous_forbidden_claims": " | ".join(grade.ambiguous_forbidden_claims),
                "rejected_forbidden_claims": " | ".join(grade.rejected_forbidden_claims),
                "total_score": total_score,
                "passed": passed,
                "rationale": grade.rationale,
                "error": "",
            })
        except Exception as exc:
            # 답변 생성 도중 실패한 경우에도 실패하기까지의 시간을 남깁니다.
            if response_seconds is None:
                response_seconds = round(time.perf_counter() - answer_started_at, 3)
            # 한 문항의 오류가 전체 평가를 중단시키지 않도록 기록 후 계속합니다.
            records.append({
                "id": test_id,
                "difficulty": row["difficulty"],
                "question": row["question"],
                "answer": "",
                "response_seconds": response_seconds,
                "conclusion_score": 0,
                "article_score": 0,
                "abstention_score": 0,
                "forbidden_claim_score": 0,
                "adopted_forbidden_claims": "",
                "ambiguous_forbidden_claims": "",
                "rejected_forbidden_claims": "",
                "total_score": 0,
                "passed": False,
                "rationale": "평가 실행 오류",
                "error": f"{type(exc).__name__}: {exc}",
            })

        latest = records[-1]
        status = "통과" if latest["passed"] else "실패"
        print(
            f"  완료 | 결론 {latest['conclusion_score']}/4"
            f" | 조문 {latest['article_score']}/2"
            f" | 조건부 판단 {latest['abstention_score']}/2"
            f" | 금지 주장 회피 {latest['forbidden_claim_score']}/2"
            f" | 총점 {latest['total_score']}/10"
            f" | 답변 {latest['response_seconds']:.3f}초 | {status}",
            flush=True,
        )
        if latest.get("adopted_forbidden_claims"):
            print(f"  채택된 금지 주장: {latest['adopted_forbidden_claims']}", flush=True)
        if latest.get("ambiguous_forbidden_claims"):
            print(f"  불분명한 금지 주장: {latest['ambiguous_forbidden_claims']}", flush=True)
        if latest["error"]:
            print(f"  오류: {latest['error']}", flush=True)
        time.sleep(delay_seconds)

    results = pd.DataFrame(records)
    if save_csv:
        output_path = TEST_DIR / f"evaluation_results_{run_id}.csv"
        results.to_csv(output_path, index=False, encoding="utf-8-sig")
        print(f"\n상세 결과 저장: {output_path}")

    valid_results = results[results["error"] == ""]
    print("\n=== 평가 요약 ===")
    print(f"평가 성공: {len(valid_results)}/{len(results)}문항")
    print(f"평균 점수: {results['total_score'].mean():.2f}/10")
    print(f"통과율: {results['passed'].mean() * 100:.1f}%")
    print(f"평균 답변 생성 시간: {results['response_seconds'].mean():.3f}초")
    if not valid_results.empty:
        summary = valid_results.groupby("difficulty").agg(
            문항수=("id", "count"),
            평균점수=("total_score", "mean"),
            평균답변시간초=("response_seconds", "mean"),
            통과율=("passed", "mean"),
        )
        summary["통과율"] = (summary["통과율"] * 100).round(1)
        display(summary.round(2))

    display(results[[
        "id", "difficulty", "response_seconds", "total_score", "passed", "rationale", "error"
    ]])
    return results


# 위 설정에 따라 선택된 문항만 순차 평가합니다.
evaluation_results = evaluate_gold_test_set(selected_gold_df)

선택된 테스트 데이터: 30개 | 난이도별 {'low': 10, 'medium': 10, 'hard': 10}


,id,difficulty,question
0,G009,low,자전거를 탄 채로 횡단보도를 건너도 돼?
1,G017,low,교차로에서 뒤에서 구급차가 오는데 그대로 진행해도 돼?
2,G001,low,빨간 신호인데 차가 한 대도 없으면 그냥 지나가도 돼?
3,G024,low,운전면허가 없는데 자동차를 잠깐만 운전해도 돼?
4,G012,low,교차로 안에서 앞차를 추월해도 돼?
5,G010,low,제한속도 50km/h 도로에서 60km/h로 달리면 위반이야?
6,G014,low,우회전할 때 미리 도로 오른쪽 가장자리에 붙지 않아도 돼?
7,G002,low,차로 보도를 조금 달려도 괜찮아?
8,G023,low,터널 안에 차를 주차해도 돼?
9,G006,low,자전거도로가 없는 도로에서 자전거는 어디로 가야 해?


[1/30] G009 Agent 질의 중...
  완료 | 결론 4/4 | 조문 2/2 | 조건부 판단 2/2 | 금지 주장 회피 1/2 | 총점 9/10 | 답변 26.634초 | 실패
  불분명한 금지 주장: 횡단보도에서는 자전거를 타고 가도 보행자와 동일하게 보호된다
[2/30] G017 Agent 질의 중...
  완료 | 결론 4/4 | 조문 2/2 | 조건부 판단 2/2 | 금지 주장 회피 2/2 | 총점 10/10 | 답변 19.708초 | 통과
[3/30] G001 Agent 질의 중...
  완료 | 결론 4/4 | 조문 2/2 | 조건부 판단 2/2 | 금지 주장 회피 2/2 | 총점 10/10 | 답변 22.152초 | 통과
[4/30] G024 Agent 질의 중...
  완료 | 결론 3/4 | 조문 1/2 | 조건부 판단 2/2 | 금지 주장 회피 1/2 | 총점 7/10 | 답변 13.789초 | 실패
  불분명한 금지 주장: 짧은 거리면 무면허운전이 아니다
[5/30] G012 Agent 질의 중...
  완료 | 결론 4/4 | 조문 1/2 | 조건부 판단 2/2 | 금지 주장 회피 1/2 | 총점 8/10 | 답변 13.420초 | 실패
  불분명한 금지 주장: 신호가 녹색이면 교차로에서도 추월할 수 있다
[6/30] G010 Agent 질의 중...
  완료 | 결론 4/4 | 조문 1/2 | 조건부 판단 2/2 | 금지 주장 회피 2/2 | 총점 9/10 | 답변 13.079초 | 통과
[7/30] G014 Agent 질의 중...
